In [38]:
from transformers import AutoImageProcessor, ResNetForImageClassification
import torch
from PIL import Image
import requests
from io import BytesIO
from datasets import load_dataset


# model_name = "microsoft/resnet-50"
# dataset = load_dataset("huggingface/cats-image")
# image = dataset["test"]["image"][0]
# image = Image.open("cat.jpg")

model_list = ["microsoft/resnet-50",
              "microsoft/resnet-18",
              "microsoft/resnet-34"]

url = "http://130.246.213.185/data/incoming/FISH/20260223/10/HDR_20260223_104545.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))
for model_name in model_list:
    print(f"model_name: {model_name}")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = ResNetForImageClassification.from_pretrained(model_name)
    inputs = processor(image, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits
    # model predicts one of the 1000 ImageNet classes
    predicted_label = logits.argmax(-1).item()
    print(model.config.id2label[predicted_label])

model_name: microsoft/resnet-50
bubble
model_name: microsoft/resnet-18
Petri dish
model_name: microsoft/resnet-34
spotlight, spot


In [10]:
# Zero-shot 分類：不用任何訓練或標記資料
# 直接告訴模型你想要的類別是什麼（用文字描述），它就會現場判斷圖片最像哪一個

from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# 這個模型（CLIP）不是固定 1000 類，而是可以現場指定任意類別
classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

# ---- 換成你自己的圖片 ----
# 情況一：本機圖片
# image = Image.open("你的圖片路徑.jpg")

# 情況二：網址圖片（例如你伺服器上的天空照片）
url = "http://130.246.213.185/data/incoming/FISH/20260223/10/HDR_20260223_104545.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# 你自己定義想分類的類別（文字描述），可以隨時修改測試不同寫法
candidate_labels = ["Cloud", "No Cloud", "Not part of the sky","Poor quality image"]

result = classifier(image, candidate_labels=candidate_labels)

print("模型判斷結果：")
for r in result:
    print(f"  {r['label']}：信心 {r['score']:.2%}")


Device set to use mps:0


模型判斷結果：
  Not part of the sky：信心 72.33%
  No Cloud：信心 23.64%
  Cloud：信心 3.52%
  Poor quality image：信心 0.52%
